# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access and display core metadata
meta = dataset.metadata
print("Dataset name:", getattr(meta, 'name', 'N/A'))
print("Description:\n", getattr(meta, 'description', 'N/A'))
print("Version:", getattr(meta, 'version', 'N/A'))
print("Date Published:", getattr(meta, 'datePublished', 'N/A'))

## 2. Data Overview

Review available record sets, fields, and their IDs.

Every record set and field in a Croissant schema has an `@id`. Here we identify included record sets and their fields.

In [ ]:
# List all available record sets and their @id
print("\nAvailable Record Sets (by @id and name):")
record_sets = []
for rs in dataset.record_sets:
    rec_id = getattr(rs, '@id', None)
    rec_name = getattr(rs, 'name', None)
    print(f"  - @id: {rec_id}, name: {rec_name}")
    record_sets.append(rec_id)
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    # For the demonstration, select the first record set
    first_rs = dataset.record_sets[0]
    print("\nFields in the first record set:")
    for field in getattr(first_rs, 'fields', []):
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"    - @id: {field_id}, name: {field_name}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract data from all record sets
recset_ids = [getattr(rs, '@id', None) for rs in dataset.record_sets]
dataframes = {}

for rec_id in recset_ids:
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"\nFirst 3 rows for record set @id: {rec_id}")
        display(df.head(3))
    else:
        print(f"No records found for record set @id: {rec_id}")

if len(dataframes) > 0:
    chosen_record_set_id = list(dataframes.keys())[0]  # select first available for next steps
    print("Columns available in DataFrame:")
    print(dataframes[chosen_record_set_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings

# Use the first numeric field found for demonstration
df = dataframes[chosen_record_set_id]
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) == 0:
    print("No numeric columns found for EDA in record set @id: ", chosen_record_set_id)
else:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field '{numeric_field}' for EDA.")
    
    # Filter (for demo, values > median)
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered to records where {numeric_field} > {threshold} (median):")
    display(filtered_df.head())

    # Normalization
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try simple grouping by a non-numeric column (if present)
    group_cols = [col for col in df.columns if df[col].dtype == 'object' and not (df[col].isnull().all())]
    if group_cols:
        group_field = group_cols[0]
        print(f"Grouping by categorical field '{group_field}'.")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical column found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Below is a histogram of the selected numeric field and a boxplot per group (if grouping is available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_cols) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot per group if group_field defined
    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric columns for visualization.")

## 6. Conclusion

This notebook demonstrated how to load, inspect, and analyze a dataset described by a Croissant schema using the `mlcroissant` library. You can adjust the filtering and visualizations to further explore specific relationships of interest for downstream analysis.